# Análise Exploratória das Métricas Locais de Conectividade

Este notebook apresenta a análise exploratória das métricas locais (por nó)
extraídas das redes funcionais cerebrais obtidas a partir de sinais de EEG
em estado de repouso.

Diferentemente das métricas globais, as métricas locais permitem investigar
a heterogeneidade topológica entre diferentes regiões cerebrais, sendo
potencialmente mais informativas para tarefas de caracterização de grupos
e aprendizado de máquina.

Objetivos desta etapa:
- analisar a distribuição das métricas locais por nó;
- investigar diferenças entre condições experimentais (EO vs EC);
- avaliar a variabilidade intersujeitos;
- preparar as features para a etapa de redução de dimensionalidade (PCA).


In [1]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_dark"

### Carregando o dataset

In [2]:
DATASET_PATH = "../../data/processed/dataset_features.parquet"

df = pd.read_parquet(DATASET_PATH)

df.shape

(108, 360)

In [3]:
df

,degree_AF3,clustering_AF3,betweenness_AF3,hub_frequency_AF3,sd_degree_AF3,bin_clustering_AF3,degree_AF4,clustering_AF4,betweenness_AF4,hub_frequency_AF4,...,age_60-65,age_65-70,age_70-75,age_75-80,gender_2,handedness_left,handedness_right,education_gymnasium,education_realschule,relationship_status_yes
0,0.421064,0.389635,0.001299,0.003559,0.185907,0.658822,0.434218,0.400541,0.000000,0.002542,...,False,False,False,False,True,False,True,True,False,False
1,0.456114,0.424352,0.000000,0.001017,0.197066,0.707606,0.447632,0.415106,0.003133,0.001017,...,False,False,False,False,True,False,True,True,False,False
2,0.585267,0.572429,0.000000,0.008134,0.221862,0.709774,0.566095,0.558080,0.000000,0.004067,...,False,False,False,False,True,False,True,False,True,True
3,0.440010,0.433551,0.000565,0.004067,0.209431,0.669874,0.422954,0.421807,0.000000,0.003559,...,False,False,False,False,True,False,True,False,True,True
4,0.665150,0.640770,0.000000,0.000000,0.182402,0.793544,0.675006,0.647611,0.000000,0.000000,...,False,False,False,False,False,True,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,0.413301,0.418309,0.000000,0.005592,0.220927,0.656618,0.415076,0.417131,0.004040,0.000000,...,False,False,False,False,True,True,False,True,False,True
104,0.465274,0.486956,0.000000,0.000000,0.225301,0.734458,0.368751,0.420833,0.000000,0.001017,...,False,False,False,False,True,False,True,True,False,False
105,0.428436,0.446453,0.000565,0.001525,0.224430,0.666602,0.389714,0.413874,0.001130,0.008643,...,False,False,False,False,True,False,True,True,False,False
106,0.608886,0.605972,0.000000,0.000508,0.233453,0.760082,0.581683,0.589138,0.000000,0.000508,...,False,False,False,False,True,False,True,True,False,True


### métricas globais × métricas locais

In [4]:
GLOBAL_FEATURES = [
    "edges_mean",
    "edges_std",
    "edges_cv"
]

NODE_FEATURE_PREFIXES = [
    "degree_",
    "clustering_",
    "betweenness_",
    "hub_frequency_",
]

In [5]:
NODE_FEATURES = [
    c for c in df.columns
    if any(c.startswith(p) for p in NODE_FEATURE_PREFIXES)
]

In [6]:
len(NODE_FEATURES)

212

DataFrame apenas com métricas locais

In [7]:
df_node = df[["subject_id", "condition"] + NODE_FEATURES].copy()
df_node.shape

(108, 214)

Distribuição de Métricas Locais

In [8]:
degree_cols = [c for c in NODE_FEATURES if c.startswith("degree_")]

df_degree_long = df_node.melt(
    id_vars=["subject_id", "condition"],
    value_vars=degree_cols,
    var_name="node",
    value_name="degree"
)

In [9]:
df_degree_long

,subject_id,condition,node,degree
0,sub-010069,EC,degree_AF3,0.421064
1,sub-010069,EO,degree_AF3,0.456114
2,sub-010070,EC,degree_AF3,0.585267
3,sub-010070,EO,degree_AF3,0.440010
4,sub-010079,EC,degree_AF3,0.665150
...,...,...,...,...
5719,sub-010315,EO,degree_TP8,0.532089
5720,sub-010316,EC,degree_TP8,NaN
5721,sub-010316,EO,degree_TP8,0.612972
5722,sub-010318,EC,degree_TP8,0.684558


In [10]:
fig = px.box(
    df_degree_long,
    x="node",
    y="degree",
    color="condition",
    title="Distribuição do Grau por Nó e Condição",
    points=False
)
fig.update_layout(xaxis_tickangle=45)
fig.show()

### Variabilidade intersujeitos (importante para ML)

In [11]:
degree_stats = (
    df_degree_long
    .groupby(["node", "condition"])["degree"]
    .agg(["mean", "std"])
    .reset_index()
)

In [12]:
fig = px.bar(
    degree_stats,
    x="node",
    y="std",
    color="condition",
    barmode="group",
    title="Variabilidade Inter-Sujeitos do Grau por Nó"
)
fig.update_layout(xaxis_tickangle=45)
fig.show()

### Heatmap sujeito × nó (padrões globais)

In [13]:
df_degree_matrix = df_node[degree_cols].copy()
df_degree_matrix.index = df_node["subject_id"]

In [14]:
fig = px.imshow(
    df_degree_matrix,
    aspect="auto",
    title="Heatmap do Grau por Nó (Sujeito × Nó)",
    color_continuous_scale="Viridis"
)
fig.show()

df_degree_long.groupby(["node", "condition"])["degree"].describe()


In [15]:
df_degree_long.groupby(["node", "condition"])["degree"].describe()

count      mean       std       min       25%       50%  \
node       condition                                                            
degree_AF3 EC          54.0  0.512514  0.105346  0.273798  0.429531  0.517561   
           EO          54.0  0.476726  0.116043  0.214845  0.417844  0.483708   
degree_AF4 EC          54.0  0.502031  0.112953  0.224868  0.413160  0.505140   
           EO          54.0  0.469721  0.120036  0.183261  0.405819  0.470246   
degree_AF8 EC          54.0  0.526350  0.113216  0.307661  0.422769  0.527471   
...                     ...       ...       ...       ...       ...       ...   
degree_POz EO          54.0  0.625549  0.080732  0.441832  0.572084  0.629665   
degree_Pz  EC          54.0  0.639985  0.068265  0.476950  0.603796  0.649819   
           EO          54.0  0.618579  0.085430  0.424444  0.562604  0.626709   
degree_TP8 EC          53.0  0.626355  0.098891  0.403876  0.571878  0.635231   
           EO          53.0  0.584108  0.116083  0.350648  0.517353  0.589636   

                           75%       max  
node       condition                      
degree_AF3 EC         0.581454  0.734681  
           EO         0.530818  0.724165  
degree_AF4 EC         0.576728  0.738909  
           EO         0.529747  0.761159  
degree_AF8 EC         0.604158  0.782088  
...                        ...       ...  
degree_POz EO         0.664025  0.841213  
degree_Pz  EC         0.684392  0.778377  
           EO         0.669705  0.816828  
degree_TP8 EC         0.685799  0.796731  
           EO         0.640764  0.817514  

[106 rows x 8 columns]